In [ ]:
#default_exp match

In [ ]:
#export
def bootstrap_matching(sQTLsMaxPips, subset, num_trials, sum_or_mean, column_annotation_based, subset_within_column):
    
    subset_freq_in_matched_set=[]
    
    final_list=[]
    
    for i in range(0, num_trials):   

        if column_annotation_based==False:
            
            matched=match_for_gc(sQTLsMaxPips, '10bp_window_gc', 0.01, 0.95)
            
        elif column_annotation_based==True:
            
            sQTLsMaxPips=sQTLsMaxPips[sQTLsMaxPips[subset]==subset_within_column]
            
            matched=match_for_gc(sQTLsMaxPips, '10bp_window_gc', 0.01, 0.95)
            
            print(sum(matched[subset]==True))
        if sum_or_mean=='sum':

            subset_freq_in_matched_set.append(sum(matched[subset]==True))
            
            sum_causal=sum(sQTLsMaxPips[sQTLsMaxPips.pip>=0.95][subset])
            sum_background=np.array(subset_freq_in_matched_set)
            
            final_list=np.log2((sum_causal+1)/(sum_background+1))
       
        elif sum_or_mean=='mean':
            subset_freq_in_matched_set.append(np.mean(matched[subset]))
            
            mean_causal=np.mean(sQTLsMaxPips[sQTLsMaxPips.pip>=0.95][subset]) 
            mean_background=np.array(subset_freq_in_matched_set)
            
            final_list=np.log2((mean_causal+1)/mean_background+1)
                
    

           
    return final_list



def plot_bootstraps(list_of_log2FEs, names_of_annot):
    
    final_df=pd.DataFrame([])
                    
    for i in range(len(list_of_log2FEs)):
    
        df = pd.DataFrame(list_of_log2FEs[i], columns=['log2_FE'])
        df['annotation']=names_of_annot[i]

        final_df=final_df.append([df])
    


    sns.set(rc={'figure.figsize':(4,8)})
    sns.set(font_scale = 2)
    sns.set_style("white")

    g = sns.catplot(
        data=final_df, kind="point",
    
        x="annotation", y="log2_FE",
        ci="sd", palette="mako", alpha=.6, height=10
        )

    g.set(ylim=(0, 7))
    g.set_xticklabels(rotation=30)